In [1]:
from transformers import AutoModel, AutoTokenizer
import torch

model = AutoModel.from_pretrained("facebook/mms-tts-eng")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-eng")

In [2]:
import os
import pandas as pd
import torchaudio
import librosa
import numpy as np
from jiwer import wer, cer

metadata = pd.read_csv("meta40.csv")
print(metadata)

           wav                                               text
0   LJ001-0001  Printing, in the only sense with which we are ...
1   LJ001-0002                     in being comparatively modern.
2   LJ001-0003  For although the Chinese took impressions from...
3   LJ001-0004  produced the block books, which were the immed...
4   LJ001-0005  the invention of movable metal letters in the ...
5   LJ001-0006  And it is worth mention in passing that, as an...
6   LJ001-0007  the earliest book printed with movable types, ...
7   LJ001-0008                          has never been surpassed.
8   LJ001-0009  Printing, then, for our purpose, may be consid...
9   LJ001-0010  Now, as all books not primarily intended as pi...
10  LJ001-0011  it is of the first importance that the letter ...
11  LJ001-0012  especially as no more time is occupied, or cos...
12  LJ001-0013        than in the same operations with ugly ones.
13  LJ001-0014  And it was a matter of course that in the Midd...
14  LJ001-

In [3]:
output_folder = "generated_wavs"
os.makedirs(output_folder, exist_ok=True)

In [7]:
import scipy
# Iterate through each row of the metadata to generate and save audio
for index, row in metadata.iterrows():
    text = row['text']
    wav_name = f"{row['wav']}.wav"  # Add the .wav extension
    output_path = os.path.join(output_folder, wav_name)

    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt")

    # Generate the audio waveform
    with torch.no_grad():
        output = model(**inputs).waveform

    # Save the generated audio as a .wav file
    scipy.io.wavfile.write(output_path, rate=model.config.sampling_rate, data=output.squeeze().numpy())

    print(f"Generated and saved: {output_path}")

Generated and saved: generated_wavs\LJ001-0001.wav
Generated and saved: generated_wavs\LJ001-0002.wav
Generated and saved: generated_wavs\LJ001-0003.wav
Generated and saved: generated_wavs\LJ001-0004.wav
Generated and saved: generated_wavs\LJ001-0005.wav
Generated and saved: generated_wavs\LJ001-0006.wav
Generated and saved: generated_wavs\LJ001-0007.wav
Generated and saved: generated_wavs\LJ001-0008.wav
Generated and saved: generated_wavs\LJ001-0009.wav
Generated and saved: generated_wavs\LJ001-0010.wav
Generated and saved: generated_wavs\LJ001-0011.wav
Generated and saved: generated_wavs\LJ001-0012.wav
Generated and saved: generated_wavs\LJ001-0013.wav
Generated and saved: generated_wavs\LJ001-0014.wav
Generated and saved: generated_wavs\LJ001-0015.wav
Generated and saved: generated_wavs\LJ001-0016.wav
Generated and saved: generated_wavs\LJ001-0017.wav
Generated and saved: generated_wavs\LJ001-0018.wav
Generated and saved: generated_wavs\LJ001-0019.wav
Generated and saved: generated_

In [13]:
import os
import csv
import librosa
import numpy as np
from jiwer import cer, wer
import parselmouth

def compute_metrics(original_folder, generated_folder, output_csv):
    metrics = {
        "File": [], "MCD": [], "LSD": [], "SNR": [],
        "Pitch RMSE": [], "Duration Difference": [], "CER": [], "WER": []
    }
    
    original_files = sorted(os.listdir(original_folder))
    generated_files = sorted(os.listdir(generated_folder))
    
    for orig_file, gen_file in zip(original_files, generated_files):
        orig_path = os.path.join(original_folder, orig_file)
        gen_path = os.path.join(generated_folder, gen_file)
        
        # Load audio files
        orig_audio, orig_sr = librosa.load(orig_path, sr=None)
        gen_audio, gen_sr = librosa.load(gen_path, sr=None)
        
        # Resample if needed
        if orig_sr != gen_sr:
            gen_audio = librosa.resample(gen_audio, gen_sr, orig_sr)
            gen_sr = orig_sr
        
        # Align audio lengths
        min_length = min(len(orig_audio), len(gen_audio))
        orig_audio = orig_audio[:min_length]
        gen_audio = gen_audio[:min_length]
        
        # Compute spectrograms
        orig_mel = librosa.feature.melspectrogram(y=orig_audio, sr=orig_sr)
        gen_mel = librosa.feature.melspectrogram(y=gen_audio, sr=gen_sr)
        
        # Align spectrogram shapes
        min_frames = min(orig_mel.shape[1], gen_mel.shape[1])
        orig_mel = orig_mel[:, :min_frames]
        gen_mel = gen_mel[:, :min_frames]
        
        # Compute metrics
        mcd = np.mean(np.abs(orig_mel - gen_mel))  # Simplified
        lsd = np.mean(np.abs(librosa.amplitude_to_db(orig_mel) - librosa.amplitude_to_db(gen_mel)))
        noise = orig_audio - gen_audio
        snr = 10 * np.log10(np.sum(orig_audio ** 2) / np.sum(noise ** 2))
        orig_pitch = parselmouth.Sound(orig_path).to_pitch().selected_array["frequency"]
        gen_pitch = parselmouth.Sound(gen_path).to_pitch().selected_array["frequency"]
        min_pitch_length = min(len(orig_pitch), len(gen_pitch))
        pitch_rmse = np.sqrt(np.mean((orig_pitch[:min_pitch_length] - gen_pitch[:min_pitch_length]) ** 2))
        duration_diff = abs(len(orig_audio) / orig_sr - len(gen_audio) / gen_sr)
        orig_text = os.path.splitext(orig_file)[0]  # Assumes filename contains transcription
        gen_text = os.path.splitext(gen_file)[0]  # Assumes filename contains transcription
        char_error_rate = cer(orig_text, gen_text)
        word_error_rate = wer(orig_text, gen_text)
        
        # Append to metrics
        metrics["File"].append(orig_file)
        metrics["MCD"].append(mcd)
        metrics["LSD"].append(lsd)
        metrics["SNR"].append(snr)
        metrics["Pitch RMSE"].append(pitch_rmse)
        metrics["Duration Difference"].append(duration_diff)
        metrics["CER"].append(char_error_rate)
        metrics["WER"].append(word_error_rate)
    
    # Write metrics to a CSV file
    with open(output_csv, mode='w', newline='') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(metrics.keys())  # Write header
        writer.writerows(zip(*metrics.values()))  # Write rows
    
    print(f"Metrics saved to {output_csv}")



# Folders containing original and generated wav files
original_folder = "D:/Wav2Lip-master/TTS Evaluation/wav"
generated_folder = "D:/Wav2Lip-master/TTS Evaluation/generated_wavs"

# Output CSV file
output_csv = "tts_model_evaluation.csv"

# Compute and save metrics
compute_metrics(original_folder, generated_folder, output_csv)


Metrics saved to tts_model_evaluation.csv
